# race90 시작 Notebook

설치와 GPU 인증은 대회 전에 README대로 준비하세요. 세 문항을 읽고 config.yaml을 수정한 뒤 아래 셀을 실행합니다. 하네스는 배경 프로세스로 실행되므로 이 Notebook에서 상태를 확인하면서 문제 풀이를 계속할 수 있습니다. 실제 대여는 실행 셀에서 시작됩니다.

In [ ]:
from pathlib import Path
import sys, json, subprocess, time
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
CONFIG = ROOT / "config.yaml"
if not CONFIG.exists():
    raise FileNotFoundError("config.example.yaml을 config.yaml로 복사하고 실제 문제에 맞게 수정하세요.")
subprocess.run([sys.executable, "-m", "race90", "plan", str(CONFIG)], cwd=ROOT, check=True)

In [ ]:
# 실제 대회 종료 시각(epoch 초)을 알면 입력하세요. None이면 시작 시점부터 90분입니다.
DEADLINE_EPOCH = None
command = [sys.executable, "-m", "race90", "run", str(CONFIG)]
if DEADLINE_EPOCH is not None:
    command += ["--deadline-epoch", str(DEADLINE_EPOCH)]
RUN_LOG = ROOT / ("launch-" + str(int(time.time())) + ".log")
with RUN_LOG.open("w") as log:
    PROCESS = subprocess.Popen(command, cwd=ROOT, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("하네스 PID:", PROCESS.pid, "로그:", RUN_LOG)

In [ ]:
# 반복 실행해서 상태를 확인하세요.
print("프로세스 종료 코드:", PROCESS.poll())
lines = RUN_LOG.read_text().splitlines()
print("\n".join(lines[-12:]))
RUN_DIR = None
for line in lines:
    try:
        event = json.loads(line)
    except json.JSONDecodeError:
        continue
    if event.get("event") == "started":
        RUN_DIR = Path(event["run_dir"])
        break
if RUN_DIR is not None:
    for filename in ["status.json", "runpod.json", "final.json"]:
        path = RUN_DIR / filename
        if path.exists():
            print(filename, path.read_text())

In [ ]:
# 완료 후 q*.solution.ipynb의 풀이 셀을 제공 Notebook에 반영하고 실행하세요.
# 제출 CSV는 아래 deliverables 폴더에 있습니다.
if RUN_DIR is not None and (RUN_DIR / "final.json").exists():
    result = json.loads((RUN_DIR / "final.json").read_text())
    print("전체 완료:", result["complete"])
    for file in sorted((RUN_DIR / "deliverables").iterdir()):
        print(file)
    print("제출 ZIP:", RUN_DIR / "submission_pack.zip")